# 02 — Busca e agregação por UF

Exemplos de consultas analíticas sobre `raizes.parquet` e `cnpj_cnaes.parquet`
usando expressões Ibis compostas (`.filter()`, `.group_by()`, `.aggregate()`).

In [ ]:
import ficha_py
from ficha_py import _

con = ficha_py.connect_ia(month="2026-04")

## Quantidade de raízes por UF

`raizes.parquet` guarda `ufs_atuacao` como array (uma raiz pode atuar em
mais de uma UF via filiais) — usamos `unnest` antes de agrupar.

In [ ]:
raizes = ficha_py.raizes(con)
(
    raizes
    .select(uf=raizes.ufs_atuacao.unnest())
    .group_by("uf")
    .aggregate(qtd_raizes=_.count())
    .order_by(_.qtd_raizes.desc())
    .limit(10)
    .execute()
)

## Empresas por CNAE, filtrado por descrição

Junta `cnpj_cnaes.parquet` (ADR 0020) com o lookup normalizado de CNAEs
(ADR 0019) para buscar por texto em vez de decorar o código.

In [ ]:
cnpj_cnaes = ficha_py.cnpj_cnaes(con)
cnaes = ficha_py.lookup_normalized(con, "cnaes")

restaurantes = (
    cnpj_cnaes
    .join(cnaes, cnpj_cnaes.cnae_codigo == cnaes.codigo)
    .filter(_.descricao_normalizada.contains("RESTAURANTE"))
    .filter(_.posicao == 0)  # só CNAE principal
    .limit(20)
)
restaurantes.execute()